In [1]:
import os
import random
import shutil
import hashlib
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.model_selection import train_test_split
from tensorflow.keras import layers, Model, Input

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

SOURCE = Path(
    "/kaggle/input/datasets/swaratakale/lineartcompletion-dip-project"
    "/CBO_Dataset_20181113/CBO_Dataset/By Item"
)

PROJECT = Path("/kaggle/working/DIP_Project")
DATASET = PROJECT / "dataset"

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp"}

PATCH_SIZE = 352
BATCH_SIZE = 4

I0000 00:00:1794932478.779228      16 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1794932493.120812      16 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


In [2]:
image_files = sorted(
    p for p in SOURCE.rglob("*")
    if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS
)

records = []

for path in image_files:
    relative = path.relative_to(SOURCE)
    category = "/".join(relative.parts[:2])

    records.append({
        "source": str(path),
        "category": category,
        "original_name": path.name
    })

df = pd.DataFrame(records)

print("Total images:", len(df))
print("Categories:", df["category"].nunique())
print(df["category"].value_counts().sort_index())

Total images: 250
Categories: 24
category
animals/butterfly    10
animals/dolphin      11
animals/elephant     11
animals/fish         11
animals/frog         10
animals/giraffe      10
animals/goat         10
animals/goose        11
animals/horse        10
animals/rabbit       11
animals/squirrel     10
animals/turtle       10
things/airplane      10
things/basketball    11
things/daisy         10
things/ice cream     10
things/kite          10
things/leaf          10
things/mitten        10
things/pear          10
things/pencil        10
things/pineapple     13
things/truck         10
things/umbrella      11
Name: count, dtype: int64


In [3]:
train_df, temp_df = train_test_split(
    df,
    test_size=0.20,
    random_state=SEED,
    stratify=df["category"]
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=SEED,
    stratify=temp_df["category"]
)

train_df = train_df.copy()
val_df = val_df.copy()
test_df = test_df.copy()

train_df["split"] = "train"
val_df["split"] = "validation"
test_df["split"] = "test"

manifest = pd.concat(
    [train_df, val_df, test_df],
    ignore_index=True
)

manifest["filename"] = manifest.apply(
    lambda r: (
        r["category"].replace("/", "__")
        + "__"
        + Path(r["original_name"]).name
    ),
    axis=1
)

print(manifest["split"].value_counts())
print("Unique filenames:", manifest["filename"].nunique())

split
train         200
validation     25
test           25
Name: count, dtype: int64
Unique filenames: 250


In [4]:
for split in ["train", "validation", "test"]:
    (DATASET / split / "Outputs").mkdir(
        parents=True, exist_ok=True
    )
    (DATASET / split / "Inputs").mkdir(
        parents=True, exist_ok=True
    )

for _, row in manifest.iterrows():
    destination = (
        DATASET / row["split"] / "Outputs" / row["filename"]
    )
    shutil.copy2(row["source"], destination)

manifest.to_csv(DATASET / "manifest.csv", index=False)

for split in ["train", "validation", "test"]:
    count = len(list((DATASET / split / "Outputs").glob("*")))
    print(f"{split}: {count} images")

train: 200 images
validation: 25 images
test: 25 images


In [5]:
train_files = list((DATASET / "train" / "Outputs").glob("*"))


def prepare_clean_patch(image_path, patch_size=PATCH_SIZE):
    image = cv2.imread(str(image_path), cv2.IMREAD_GRAYSCALE)

    h, w = image.shape
    scale = max(
        patch_size / min(h, w),
        random.uniform(0.75, 1.25)
    )

    new_w = int(w * scale)
    new_h = int(h * scale)

    image = cv2.resize(image, (new_w, new_h))

    angle = random.uniform(0, 360)
    matrix = cv2.getRotationMatrix2D(
        (new_w / 2, new_h / 2), angle, 1
    )

    image = cv2.warpAffine(
        image,
        matrix,
        (new_w, new_h),
        borderValue=255
    )

    if random.random() < 0.5:
        image = cv2.flip(image, 1)

    if random.random() < 0.5:
        image = cv2.flip(image, 0)

    h, w = image.shape
    y = random.randint(0, h - patch_size)
    x = random.randint(0, w - patch_size)

    return image[y:y + patch_size, x:x + patch_size]


def create_damaged_patch(
    clean_patch,
    min_squares=10,
    max_squares=20,
    min_side=10,
    max_side=50
):
    damaged = clean_patch.copy()
    h, w = damaged.shape

    for _ in range(random.randint(min_squares, max_squares)):
        side = random.randint(min_side, max_side)
        x = random.randint(0, w - side)
        y = random.randint(0, h - side)

        damaged[y:y + side, x:x + side] = 255

    return damaged

In [6]:
def training_generator(
    image_files,
    batch_size=BATCH_SIZE,
    patch_size=PATCH_SIZE
):
    while True:
        batch_x, batch_y = [], []

        for _ in range(batch_size):
            image_path = random.choice(image_files)

            clean = prepare_clean_patch(
                image_path, patch_size
            )
            damaged = create_damaged_patch(clean)

            batch_x.append(damaged.astype("float32") / 255.0)
            batch_y.append(clean.astype("float32") / 255.0)

        yield (
            np.array(batch_x)[..., np.newaxis],
            np.array(batch_y)[..., np.newaxis]
        )


sample_x, sample_y = next(training_generator(train_files))

print("Input shape:", sample_x.shape)
print("Target shape:", sample_y.shape)
print("Input range:", sample_x.min(), sample_x.max())

Input shape: (4, 352, 352, 1)
Target shape: (4, 352, 352, 1)
Input range: 0.0 1.0


In [7]:
def stable_seed(filename, split):
    value = f"{split}_{filename}".encode()
    return int(hashlib.sha256(value).hexdigest()[:8], 16)


def create_fixed_damaged_image(
    clean_image,
    seed,
    min_squares=10,
    max_squares=20,
    min_side=10,
    max_side=50
):
    rng = random.Random(seed)
    damaged = clean_image.copy()

    h, w = damaged.shape
    foreground = np.argwhere(clean_image < 128)

    if len(foreground) == 0:
        return damaged

    for _ in range(rng.randint(min_squares, max_squares)):
        cy, cx = foreground[
            rng.randrange(len(foreground))
        ]

        side = rng.randint(min_side, max_side)
        half = side // 2

        x1 = max(0, cx - half)
        x2 = min(w, cx + half)
        y1 = max(0, cy - half)
        y2 = min(h, cy + half)

        damaged[y1:y2, x1:x2] = 255

    return damaged


def generate_fixed_split(split):
    output_dir = DATASET / split / "Outputs"
    input_dir = DATASET / split / "Inputs"

    for output_path in sorted(output_dir.iterdir()):
        clean = cv2.imread(
            str(output_path), cv2.IMREAD_GRAYSCALE
        )

        damaged = create_fixed_damaged_image(
            clean,
            stable_seed(output_path.name, split)
        )

        cv2.imwrite(
            str(input_dir / output_path.name),
            damaged
        )

    print(
        split,
        "Inputs:", len(list(input_dir.iterdir())),
        "Outputs:", len(list(output_dir.iterdir()))
    )


generate_fixed_split("validation")
generate_fixed_split("test")

validation Inputs: 25 Outputs: 25
test Inputs: 25 Outputs: 25


In [8]:
for split in ["train", "validation", "test"]:
    outputs = {
        p.name for p in (DATASET / split / "Outputs").iterdir()
    }
    inputs = {
        p.name for p in (DATASET / split / "Inputs").iterdir()
    }

    print(f"\n{split}")
    print("Outputs:", len(outputs))
    print("Inputs:", len(inputs))

    if split != "train":
        assert inputs == outputs, "Input-output mismatch!"


train
Outputs: 200
Inputs: 0

validation
Outputs: 25
Inputs: 25

test
Outputs: 25
Inputs: 25


In [9]:
def conv_bn_relu(x, filters, kernel_size=3, strides=1):
    x = layers.Conv2D(
        filters,
        kernel_size,
        strides=strides,
        padding="same",
        use_bias=False
    )(x)

    x = layers.BatchNormalization()(x)
    return layers.ReLU()(x)


def build_model1():
    inputs = Input(
        shape=(None, None, 1),
        name="damaged_image"
    )

    x = conv_bn_relu(inputs, 24, 5, 2)
    x = conv_bn_relu(x, 64, strides=2)
    x = conv_bn_relu(x, 128)
    x = conv_bn_relu(x, 256, strides=2)
    x = conv_bn_relu(x, 512)
    x = conv_bn_relu(x, 512, strides=2)
    x = conv_bn_relu(x, 256)

    x = layers.UpSampling2D(2, interpolation="nearest")(x)
    x = conv_bn_relu(x, 128)
    x = conv_bn_relu(x, 64)

    x = layers.UpSampling2D(2, interpolation="nearest")(x)
    x = conv_bn_relu(x, 32)
    x = conv_bn_relu(x, 16)

    x = layers.UpSampling2D(2, interpolation="nearest")(x)
    x = conv_bn_relu(x, 8)
    x = conv_bn_relu(x, 4)

    x = layers.UpSampling2D(2, interpolation="nearest")(x)
    x = conv_bn_relu(x, 2)
    x = conv_bn_relu(x, 1)

    outputs = layers.Conv2D(
        1, 3, padding="same", activation="sigmoid"
    )(x)

    return Model(inputs, outputs, name="Model1_Encoder_Decoder")

In [10]:
def conv_bn_relu_dilated(
    x, filters, kernel_size=3, dilation_rate=1
):
    x = layers.Conv2D(
        filters,
        kernel_size,
        padding="same",
        dilation_rate=dilation_rate,
        use_bias=False
    )(x)

    x = layers.BatchNormalization()(x)
    return layers.ReLU()(x)


def build_model2():
    inputs = Input(
        shape=(None, None, 1),
        name="damaged_image"
    )

    x = conv_bn_relu_dilated(inputs, 64, kernel_size=11)
    x = conv_bn_relu_dilated(x, 128)
    x = conv_bn_relu_dilated(x, 256, dilation_rate=1)
    x = conv_bn_relu_dilated(x, 256, dilation_rate=2)
    x = conv_bn_relu_dilated(x, 256, dilation_rate=4)
    x = conv_bn_relu_dilated(x, 256, dilation_rate=8)
    x = conv_bn_relu_dilated(x, 128)
    x = conv_bn_relu_dilated(x, 64)

    outputs = layers.Conv2D(
        1, 3, padding="same", activation="sigmoid"
    )(x)

    return Model(inputs, outputs, name="Model2_Dilated_CNN")

In [11]:
model1 = build_model1()
model2 = build_model2()

print("MODEL 1")
model1.summary()

print("\nMODEL 2")
model2.summary()

E0000 00:00:1794932514.520582      16 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


MODEL 1


Model: "Model1_Encoder_Decoder"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ damaged_image (InputLayer)      │ (None, None, None, 1)  │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, None, None, 24) │           600 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, None, None, 24) │            96 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu (ReLU)                    │ (None, None, None, 24) │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, None, None, 64) │        13,824 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, None, None, 64) │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_1 (ReLU)                  │ (None, None, None, 64) │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, None, None,     │        73,728 │
│                                 │ 128)                   │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, None, None,     │           512 │
│ (BatchNormalization)            │ 128)                   │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_2 (ReLU)                  │ (None, None, None,     │             0 │
│                                 │ 128)                   │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, None, None,     │       294,912 │
│                                 │ 256)                   │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, None, None,     │         1,024 │
│ (BatchNormalization)            │ 256)                   │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_3 (ReLU)                  │ (None, None, None,     │             0 │
│                                 │ 256)                   │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, None, None,     │     1,179,648 │
│                                 │ 512)                   │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, None, None,     │         2,048 │
│ (BatchNormalization)            │ 512)                   │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_4 (ReLU)                  │ (None, None, None,     │             0 │
│                                 │ 512)                   │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, None, None,     │     2,359,296 │
│                                 │ 512)                   │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_5           │ (None, None, None,     │         2,048 │
│ (BatchNormalization)            │ 512)                   │               │
├─────────────────────────────────┼────────────────────────┼─────────────

 Total params: 5,502,904 (20.99 MB)

 Trainable params: 5,498,890 (20.98 MB)

... (3 outputs left out)

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ damaged_image (InputLayer)      │ (None, None, None, 1)  │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_16 (Conv2D)              │ (None, None, None, 64) │         7,744 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_15          │ (None, None, None, 64) │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_15 (ReLU)                 │ (None, None, None, 64) │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_17 (Conv2D)              │ (None, None, None,     │        73,728 │
│                                 │ 128)                   │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_16          │ (None, None, None,     │           512 │
│ (BatchNormalization)            │ 128)                   │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_16 (ReLU)                 │ (None, None, None,     │             0 │
│                                 │ 128)                   │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_18 (Conv2D)              │ (None, None, None,     │       294,912 │
│                                 │ 256)                   │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_17          │ (None, None, None,     │         1,024 │
│ (BatchNormalization)            │ 256)                   │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_17 (ReLU)                 │ (None, None, None,     │             0 │
│                                 │ 256)                   │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_19 (Conv2D)              │ (None, None, None,     │       589,824 │
│                                 │ 256)                   │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_18          │ (None, None, None,     │         1,024 │
│ (BatchNormalization)            │ 256)                   │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_18 (ReLU)                 │ (None, None, None,     │             0 │
│                                 │ 256)                   │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_20 (Conv2D)              │ (None, None, None,     │       589,824 │
│                                 │ 256)                   │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_19          │ (None, None, None,     │         1,024 │
│ (BatchNormalization)            │ 256)                   │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_19 (ReLU)                 │ (None, None, None,     │             0 │
│                                 │ 256)                   │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_21 (Conv2D)              │ (None, None, None,     │       589,824 │
│                                 │ 256)                   │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_20          │ (None, None, None,     │         1,02

 Total params: 2,520,705 (9.62 MB)

 Trainable params: 2,517,889 (9.60 MB)

 Non-trainable params: 2,816 (11.00 KB)

In [12]:
print("Available GPUs:", tf.config.list_physical_devices("GPU"))

x = tf.random.uniform((1, PATCH_SIZE, PATCH_SIZE, 1))

for name, model in [
    ("Model 1", model1),
    ("Model 2", model2)
]:
    prediction = model(x, training=False)

    print(
        name,
        "Input:", x.shape,
        "Output:", prediction.shape,
        "Parameters:", model.count_params()
    )

Available GPUs: []
Model 1 Input: (1, 352, 352, 1) Output: (1, 352, 352, 1) Parameters: 5502904
Model 2 Input: (1, 352, 352, 1) Output: (1, 352, 352, 1) Parameters: 2520705


In [13]:

from pathlib import Path

roots = [
    Path("/kaggle/working"),
    Path("/kaggle/input/datasets/swaratakale")
]

keywords = [
    "test", "input", "groundtruth", "ground_truth",
    "outputs", "prediction", "predictions", "result",
    "manifest", "line_completion_eval", "dip_project"
]

image_exts = {".png", ".jpg", ".jpeg", ".bmp"}

print("========== RELEVANT DIRECTORIES ==========")

for root in roots:
    print(f"\nROOT: {root}")
    if not root.exists():
        print("Not found")
        continue

    for p in root.rglob("*"):
        if p.is_dir() and any(k in p.name.lower() for k in keywords):
            try:
                count = sum(
                    1 for f in p.iterdir()
                    if f.is_file() and f.suffix.lower() in image_exts
                )
                print(f"{p}  |  {count} direct images")
            except PermissionError:
                pass

print("\n========== MANIFEST FILES ==========")

for root in roots:
    if root.exists():
        for p in root.rglob("*"):
            if p.is_file() and (
                "manifest" in p.name.lower()
                or p.suffix.lower() == ".csv"
            ):
                print(p)


========== RELEVANT DIRECTORIES ==========

ROOT: /kaggle/working
/kaggle/working/DIP_Project  |  0 direct images
/kaggle/working/DIP_Project/dataset/test  |  0 direct images
/kaggle/working/DIP_Project/dataset/validation/Inputs  |  25 direct images
/kaggle/working/DIP_Project/dataset/validation/Outputs  |  25 direct images
/kaggle/working/DIP_Project/dataset/train/Inputs  |  0 direct images
/kaggle/working/DIP_Project/dataset/train/Outputs  |  200 direct images
/kaggle/working/DIP_Project/dataset/test/Inputs  |  25 direct images
/kaggle/working/DIP_Project/dataset/test/Outputs  |  25 direct images

ROOT: /kaggle/input/datasets/swaratakale
/kaggle/input/datasets/swaratakale/lineartcompletion-dip-project/Inputs  |  37 direct images
/kaggle/input/datasets/swaratakale/lineartcompletion-dip-project/Inputs_For_Prediction  |  10 direct images
/kaggle/input/datasets/swaratakale/lineartcompletion-dip-project/Outputs  |  37 direct images
/kaggle/input/datasets/swaratakale/lineartcompletion-dip-

In [14]:

from pathlib import Path
import pandas as pd

BASE = Path("/kaggle/working/DIP_Project/dataset")
TEST_IN = BASE / "test/Inputs"
TEST_OUT = BASE / "test/Outputs"
MANIFEST = BASE / "manifest.csv"

print("========== TEST FILE PAIRS ==========")

inputs = sorted(p.name for p in TEST_IN.iterdir() if p.is_file())
outputs = sorted(p.name for p in TEST_OUT.iterdir() if p.is_file())

print("Input count:", len(inputs))
print("Output count:", len(outputs))
print("Same filenames:", inputs == outputs)

print("\nSample filenames:")
for name in inputs[:10]:
    print(name)

print("\n========== MANIFEST ==========")
df = pd.read_csv(MANIFEST)
print("Columns:", df.columns.tolist())
print(df.head(10).to_string(index=False))

print("\n========== PREDICTION WORKSPACE ==========")
WORK = Path("/kaggle/working/line_completion_eval")

if WORK.exists():
    for p in sorted(WORK.rglob("*")):
        if p.is_dir():
            files = [f for f in p.iterdir() if f.is_file()]
            if files:
                print(f"{p} | {len(files)} files | examples: {[f.name for f in files[:3]]}")
else:
    print("The proposed evaluation workspace is not present at this path.")


========== TEST FILE PAIRS ==========
Input count: 25
Output count: 25
Same filenames: True

Sample filenames:
animals__butterfly__butterfly6.jpg
animals__dolphin__dolphin6.jpg
animals__elephant__elephant3.jpg
animals__fish__fish8.jpg
animals__frog__frog6.jpg
animals__giraffe__giraffe4.jpg
animals__goat__goat1.jpg
animals__goose__goose2.jpg
animals__horse__horse7.jpg
animals__rabbit__rabbit10.jpg

========== MANIFEST ==========
Columns: ['source', 'category', 'original_name', 'split', 'filename']
                                                                                                                                   source         category  original_name split                          filename
      /kaggle/input/datasets/swaratakale/lineartcompletion-dip-project/CBO_Dataset_20181113/CBO_Dataset/By Item/things/pencil/pencil5.jpg    things/pencil    pencil5.jpg train       things__pencil__pencil5.jpg
         /kaggle/input/datasets/swaratakale/lineartcompletion-dip-project/CBO_

In [15]:

import pandas as pd
from pathlib import Path

manifest = pd.read_csv(
    "/kaggle/working/DIP_Project/dataset/manifest.csv"
)

test_df = manifest[manifest["split"] == "test"].copy()
test_df = test_df.sort_values("filename")

test_filenames = test_df["filename"].tolist()

print("Test images:", len(test_filenames))
print("\n".join(test_filenames))

assert len(test_filenames) == 25
assert len(set(test_filenames)) == 25


Test images: 25
animals__butterfly__butterfly6.jpg
animals__dolphin__dolphin6.jpg
animals__elephant__elephant3.jpg
animals__fish__fish8.jpg
animals__frog__frog6.jpg
animals__giraffe__giraffe4.jpg
animals__goat__goat1.jpg
animals__goose__goose2.jpg
animals__horse__horse7.jpg
animals__rabbit__rabbit10.jpg
animals__squirrel__squirrel8.jpg
animals__turtle__turtle2.JPG
things__airplane__airplane8.jpg
things__basketball__basketball5.jpg
things__daisy__daisy2.jpg
things__ice cream__icecream9.jpg
things__kite__kite6.jpg
things__leaf__leaf2.jpg
things__mitten__mitten4.jpg
things__pear__pear6.jpg
things__pencil__pencil3.jpg
things__pineapple__pineapple11.jpg
things__pineapple__pineapple6.jpg
things__truck__truck6.jpg
things__umbrella__umbrella10.jpg
